In [ ]:
import numpy as np
import scipy
from scipy.io.wavfile import read
import matplotlib.pyplot as plt
from IPython.display import Audio
import librosa, librosa.display
plt.rcParams['figure.figsize'] = (8, 4)
from scipy import signal
from librosa import feature, frames_to_time, autocorrelate, clicks

# Week 8 Activity: Onset Detection and Autocorrelation

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code.

## Onset Detection

Read in the following files below, and normalize.

a) 80sPopDrums.wav  
b) TheBlackKeys_track4.wav (use random 10s from near middle of track)  
c) prelude_cmaj_10s.wav  

In [ ]:
(fs_a, a) = read('../Audio/80sPopDrums.wav')
dur_a = a.size/fs_a
time_a = np.arange(0, dur_a, 1/fs_a)
anorm = a/np.abs(a.max())

(fs_b, b) = read('../Audio/TheBlackKeys_track4.wav')
# slice audio to be from 10s - 20s
b = b[(10 * fs_b):(20* fs_b)]
dur_b = b.size/fs_b
time_b = np.arange(0, dur_b, 1/fs_b)
bnorm = b/np.abs(b.max())

(fs_c, c) = read('../audio/prelude_cmaj_10s.wav')
if c.ndim == 2:
    c = c.mean(axis=1) 

dur_c = c.size/fs_c
time_c = np.arange(0, dur_c, 1/fs_c)
cnorm = c/np.abs(c.max())

#### For each of the tracks (start with just one):

1) Compute the energy and RMSE for each track using a frame length of your choice  
2) Plot the energy and the RMSE
3) Use one of the energy features to create your novelty function and plot
4) Write a peak picking function to define onsets from your novelty function
4) Based on the plots, estimate what an appropriate 'threshold' would be. Does the threshold change across tracks? If so, what could you do to put them on a similar scale?

In [ ]:
# start with a

hop_length = 512
frame_length = 1024

energy_a = np.array([
    sum(abs(anorm[i:i+frame_length]**2)) for i in range(0, len(anorm), hop_length)
])

# energy_a.shape
# energy_a[0]

rmse_a = librosa.feature.rms(y = anorm, frame_length=frame_length, hop_length=hop_length, center=True)

# rmse_a.shape

frames_a = range(len(energy_a))

t_a = librosa.frames_to_time(frames_a, sr=fs_a, hop_length=hop_length)

librosa.display.waveshow(anorm, sr=fs_a, alpha=0.5)
plt.plot(t_a, energy_a/energy_a.max(), 'r', label='energy')
plt.plot(t_a, rmse_a[0]/rmse_a[0].max(), color='b', label='rmse')
plt.title('normalized signal (a) w/ energy and rms overlaid')
plt.legend()

In [ ]:
rmse_diff_a = np.diff(rmse_a[0])
novelty_a = np.concatenate((rmse_diff_a, np.array([0])))
novelty_a[novelty_a < 0] = 0
plt.plot(t_a, novelty_a/novelty_a.max(), label='normalized novelty function')
plt.xlabel('time (s)')
plt.title('normalized novelty function from rmse')
plt.legend()
plt.show()

In [ ]:
# peak picking
novelty_a_peaks = np.where(novelty_a > 0.1)[0]
peak_times_a = t_a[novelty_a_peaks]

plt.plot(t_a, novelty_a)
plt.vlines(peak_times_a, 0, np.max(novelty_a), colors='r')

In [ ]:
threshold_a = 0.1
peak_indices_a = []

for i in range(1, len(novelty_a)-1):
    if novelty_a[i] > threshold_a:
        if novelty_a[i] > novelty_a[i-1] and novelty_a[i] > novelty_a[i+1]:
            peak_indices_a.append(i)

peak_indices_a = np.array(peak_indices_a)
peak_times_a = t_a[peak_indices_a]

plt.plot(t_a, novelty_a, label='novelty function')
plt.vlines(peak_times_a, 0, np.max(novelty_a), colors='r')
plt.xlabel('time (s)')
plt.title('novelty function with threshold peaks')
plt.legend()
plt.show()

In [ ]:
# b

energy_b = np.array([
    sum(abs(bnorm[i:i+frame_length]**2)) for i in range(0, len(bnorm), hop_length)
])

rmse_b = librosa.feature.rms(y = bnorm, frame_length=frame_length, hop_length=hop_length, center=True)

frames_b = range(len(energy_b))

t_b = librosa.frames_to_time(frames_b, sr=fs_b, hop_length=hop_length)

librosa.display.waveshow(bnorm, sr=fs_b, alpha=0.5)
plt.plot(t_b, energy_b/energy_b.max(), 'r', label='energy')
plt.plot(t_b, rmse_b[0]/rmse_b[0].max(), color='b', label='rmse')
plt.title('normalized signal (b) w/ energy and rms overlaid')
plt.legend()

In [ ]:
rmse_diff_b = np.diff(rmse_b[0])
novelty_b = np.concatenate((rmse_diff_b, np.array([0])))
novelty_b[novelty_b < 0] = 0
plt.plot(t_b, novelty_b/novelty_b.max(), label='normalized novelty function')
plt.xlabel('time (s)')
plt.title('normalized novelty function from rmse')
plt.legend()
plt.show()

In [ ]:
# peak picking
novelty_b_peaks = np.where(novelty_b > 0.1)[0]
peak_times_b = t_b[novelty_b_peaks]

plt.plot(t_b, novelty_b)
plt.vlines(peak_times_b, 0, np.max(novelty_b), colors='r')

In [ ]:
threshold_b = 0.1
peak_indices_b = []

for i in range(1, len(novelty_b)-1):
    if novelty_b[i] > threshold_b:
        if novelty_b[i] > novelty_b[i-1] and novelty_b[i] > novelty_b[i+1]:
            peak_indices_b.append(i)

peak_indices_b = np.array(peak_indices_b)
peak_times_b = t_b[peak_indices_b]

plt.plot(t_b, novelty_b, label='novelty function')
plt.vlines(peak_times_b, 0, np.max(novelty_b), colors='r')
plt.xlabel('time (s)')
plt.title('novelty function with threshold peaks')
plt.legend()
plt.show()

In [ ]:
# c
hop_length_c = 256
frame_length_c = 512

energy_c = np.array([
    sum(abs(cnorm[i:i+frame_length_c]**2)) for i in range(0, len(cnorm), hop_length_c)
])


energy_c.shape
energy_c[0]

rmse_c = librosa.feature.rms(y = cnorm, frame_length=frame_length_c, hop_length=hop_length_c, center=True)

rmse_c.shape

frames_c = range(len(energy_c))

t_c = librosa.frames_to_time(frames_c, sr=fs_c, hop_length=hop_length_c)

rmse_diff_c = np.diff(rmse_c[0])
novelty_c = np.concatenate((rmse_diff_c, np.array([0])))
novelty_c[novelty_c < 0] = 0

novelty_c_peaks = np.where(novelty_c > 0.1)[0]
peak_times_c = t_c[novelty_c_peaks]

threshold_c = 0.08
peak_indices_c = []

for i in range(1, len(novelty_c)-1):
    if novelty_c[i] > threshold_c:
        if novelty_c[i] > novelty_c[i-1] and novelty_c[i] > novelty_c[i+1]:
            peak_indices_c.append(i)

peak_indices_c = np.array(peak_indices_c)
peak_times_c = t_c[peak_indices_c]

plt.plot(t_c, novelty_c, label='novelty function')
plt.vlines(peak_times_c, 0, np.max(novelty_c), colors='r')
plt.xlabel('time (s)')
plt.title('novelty function with threshold peaks')
plt.legend()
plt.show()

## Tempo Estimation

Using your novelty functions and detected onsets above, estimate the global tempo of the files using autocorrelation and IOIs.

1) IOI Based Tempo

- Compute Inter-Onset Intervals (IOIs)
- Convert IOIs to BPM
- Create a beat histogram to help you infer tempos

In [ ]:
from collections import Counter

In [ ]:
# a
peaks_a, properties_a = signal.find_peaks(novelty_a, height=0.1)
peak_times_a = t_a[1:][peaks_a]

iois_a = np.diff(peak_times_a)
num_counts_a = Counter(iois_a)
top_n_a = num_counts_a.most_common(3)

print(top_n_a)

tempo1_a = 60/0.546
tempo2_a = 60/0.128
tempo3_a = 60/0.139

print(tempo1_a, tempo2_a, tempo3_a)

ac_a = librosa.autocorrelate(novelty_a)
lags_a = np.arange(len(ac_a))
lag_times_a = lags_a * hop_length / fs_a
bpms_a = 60 / lag_times_a[1:]   # avoid divide by zero
strength_a = ac_a[1:]

plt.plot(bpms_a, strength_a)
plt.xlim(40,300)
plt.ylabel('bpm')
plt.ylabel('beat strength')
plt.title('beat histogram')

In [ ]:
# b
peaks_b, properties_b = signal.find_peaks(novelty_b, height=0.1)
peak_times_b = t_b[1:][peaks_b]

iois_b = np.diff(peak_times_b)
num_counts_b = Counter(iois_b)
top_n_b = num_counts_b.most_common(3)

print(top_n_b)

tempo1_b = 60/0.221
tempo2_b = 60/0.081
tempo3_b = 60/0.209

print(tempo1_b, tempo2_b, tempo3_b)


ac_b = librosa.autocorrelate(novelty_b)
lags_b = np.arange(len(ac_b))
lag_times_b = lags_b * hop_length / fs_b
bpms_b = 60 / lag_times_b[1:]   # avoid divide by zero
strength_b = ac_b[1:]

plt.plot(bpms_b, strength_b)
plt.xlim(40,300)
plt.ylabel('bpm')
plt.ylabel('beat strength')
plt.title('beat histogram')

In [ ]:
# c
peaks_c, properties_c = signal.find_peaks(novelty_c, height=0.1)
peak_times_c = t_c[1:][peaks_c]

iois_c = np.diff(peak_times_c)
num_counts_c = Counter(iois_c)
top_n_c = num_counts_c.most_common(3)

print(top_n_c)

tempo1_c = 60/0.221
tempo2_c = 60/0.081
tempo3_c = 60/0.209

print(tempo1_c, tempo2_c, tempo3_c)


ac_c = librosa.autocorrelate(novelty_c)
lags_c = np.arange(len(ac_c))
lag_times_c = lags_c * hop_length / fs_c
bpms_c = 60 / lag_times_c[1:]   # avoid divide by zero
strength_c = ac_c[1:]

plt.plot(bpms_c, strength_c)
plt.xlim(40,300)
plt.ylabel('bpm')
plt.ylabel('beat strength')
plt.title('beat histogram')

2) Autocorrelation based Tempo
- Compute the autocorrelation of your novelty function
- Convert lag to BPMs (conside perceptual relevance)
- Crate a beat histogram to help you infer tempos

In [ ]:
#a
ac_a = librosa.autocorrelate(novelty_a)
ac_a = ac_a / ac_a.max()

plt.figure(figsize=(12,5))
plt.plot(ac_a)
plt.title('Autocorrelation of Energy differential')
plt.xlabel('Time lag (in frames)')

In [ ]:
lags_a = np.arange(len(ac_a))

plt.plot(lags_a * hop_length / fs_a, ac_a)
plt.xlabel("Lag (in seconds)")
plt.title("Autocorrelation of Energy Differential")

In [ ]:
ac_a[1:].argmax()

In [ ]:
points_a = np.where(ac_a > 0.3)
points_a

In [ ]:
points_a = np.array(points_a)
t_inc_a = hop_length * points_a / fs_a
# t_inc_a

In [ ]:
ts_a = t_inc_a[0][2:]
durs_a = ts_a[1:] - ts_a[:-1]
durs_a = np.round(durs_a, 3)
durs_a

In [ ]:
num_counts_a = Counter(durs_a)
top_n_a = num_counts_a.most_common(3)
print(top_n_a)

In [ ]:
atem1 = 60 / top_n_a[0][0]
atem2 = 60 / top_n_a[1][0]
atem3 = 60 / top_n_a[2][0]

print(atem1, atem2, atem3)

In [ ]:
plt.plot(bpms_a, strength_a)
plt.xlim(40, 300)
plt.xlabel("Tempo (BPM)")
plt.ylabel("Beat Strength")
plt.title("Beat Histogram (Autocorrelation)")

In [ ]:
#b

ac_b = librosa.autocorrelate(novelty_b)
ac_b = ac_b / ac_b.max()

lags_b = np.arange(len(ac_b))

points_b = np.where(ac_b > 0.3)

points_b = np.array(points_b)
t_inc_b = hop_length * points_b / fs_b

ts_b = t_inc_b[0][2:]
durs_b = ts_b[1:] - ts_b[:-1]
durs_b = np.round(durs_b, 3)

num_counts_b = Counter(durs_b)
top_n_b = num_counts_b.most_common(3)

btem1 = 60 / top_n_b[0][0]
btem2 = 60 / top_n_b[1][0]
btem3 = 60 / top_n_b[2][0]

print(btem1, btem2, btem3)



In [ ]:
plt.plot(bpms_b, strength_b)
plt.xlim(40, 300)
plt.xlabel("Tempo (BPM)")
plt.ylabel("Beat Strength")
plt.title("Beat Histogram (Autocorrelation)")

In [ ]:
#b

ac_c = librosa.autocorrelate(novelty_c)
ac_c = ac_c / ac_c.max()

lags_c = np.arange(len(ac_c))

points_c = np.where(ac_c > 0.3)

points_c = np.array(points_c)
t_inc_c = hop_length * points_c / fs_c

ts_c = t_inc_c[0][2:]
durs_c = ts_c[1:] - ts_c[:-1]
durs_c = np.round(durs_c, 3)

num_counts_c = Counter(durs_c)
top_n_c = num_counts_c.most_common(3)

ctem1 = 60 / top_n_c[0][0]
ctem2 = 60 / top_n_c[1][0]
ctem3 = 60 / top_n_c[2][0]

print(ctem1, ctem2, ctem3)

In [ ]:
plt.plot(bpms_c, strength_c)
plt.xlim(40, 300)
plt.xlabel("Tempo (BPM)")
plt.ylabel("Beat Strength")
plt.title("Beat Histogram (Autocorrelation)")


Estimate tempo using librosa.beat.tempo. How do your results compare?

Between autocorrelation and IOIs, which method is most stable? Which is most sensitive to onset errors Do the different files present different behaviors/accuracies?

In [ ]:
print(librosa.beat.tempo(novelty_a))
print(librosa.beat.tempo(novelty_b))
print(librosa.beat.tempo(novelty_c))

IOI Based detection was wildly off for b and c, whereas autocorrelation was more accurate. None of the results match the output of the librosa function.

## Extension

Repeat the activities above but change the frame size and hop length. How does that impact your results?

In [ ]:
# a with a quarter the frame size and hop length
hop_length = 128
frame_length = 256

(fs_a, a) = read('../Audio/80sPopDrums.wav')
dur_a = a.size/fs_a
time_a = np.arange(0, dur_a, 1/fs_a)
anorm = a/np.abs(a.max())

energy_a = np.array([
    sum(abs(anorm[i:i+frame_length]**2)) for i in range(0, len(anorm), hop_length)
])


energy_a.shape
energy_a[0]

rmse_a = librosa.feature.rms(y = anorm, frame_length=frame_length, hop_length=hop_length, center=True)

rmse_a.shape

frames_a = range(len(energy_a))

t_a = librosa.frames_to_time(frames_a, sr=fs_a, hop_length=hop_length)

rmse_diff_a = np.diff(rmse_a[0])
novelty_a = np.concatenate((rmse_diff_a, np.array([0])))
novelty_a[novelty_a < 0] = 0

novelty_a_peaks = np.where(novelty_a > 0.1)[0]
peak_times_a = t_a[novelty_a_peaks]

threshold_a = 0.08
peak_indices_a = []

for i in range(1, len(novelty_a)-1):
    if novelty_a[i] > threshold_a:
        if novelty_a[i] > novelty_a[i-1] and novelty_a[i] > novelty_a[i+1]:
            peak_indices_a.append(i)

peak_indices_a = np.array(peak_indices_a)
peak_times_a = t_a[peak_indices_a]

plt.plot(t_a, novelty_a, label='novelty function')
plt.vlines(peak_times_a, 0, np.max(novelty_a), colors='r')
plt.xlabel('time (s)')
plt.title('novelty function with threshold peaks')
plt.legend()
plt.show()

In [ ]:
# a
peaks_a, properties_a = signal.find_peaks(novelty_a, height=0.1)
peak_times_a = t_a[1:][peaks_a]

iois_a = np.diff(peak_times_a)
num_counts_a = Counter(iois_a)
top_n_a = num_counts_a.most_common(3)

print(top_n_a)

tempo1_a = 60/0.546
tempo2_a = 60/0.128
tempo3_a = 60/0.139

print(tempo1_a, tempo2_a, tempo3_a)

ac_a = librosa.autocorrelate(novelty_a)
lags_a = np.arange(len(ac_a))
lag_times_a = lags_a * hop_length / fs_a
bpms_a = 60 / lag_times_a[1:]   # avoid divide by zero
strength_a = ac_a[1:]

plt.plot(bpms_a, strength_a)
plt.xlim(40,300)
plt.ylabel('bpm')
plt.ylabel('beat strength')
plt.title('beat histogram')

shrinking frame/hop size seems to only have created a more precise beat histogram; the resultant likely tempos remain the same